# U-Net — Football Semantic Segmentation

**Dataset:** [Football (Semantic Segmentation)](https://www.kaggle.com/datasets/sadhliroomyprime/football-semantic-segmentation)  
**Reference notebook:** https://www.kaggle.com/code/mikhaylov1007/footbal-segmentate

### Classes and file format
The dataset has 11 annotated classes: Goal Bar, Referee, Advertisements, Ground,
Ball, Coaches & Officials, Audience, Goalkeeper A, Goalkeeper B, Team A, and Team B.
Class names and category IDs are read from `COCO_Football Pixel.json` in Cell 4.
Unannotated pixels are treated as an additional Background class.

Original images and exported masks share the `images/` directory. Fused masks use
the suffix `___fuse.png`; `___save.png` files encode instances. The exported mask
colors differ from the COCO category colors, so training labels are reconstructed
from COCO segmentation geometry using `pycocotools`, then saved as class-index PNGs.
Plots use a separate display palette.

### Steps
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`.
2. Run **Cell 1** to install packages.
3. Run **Cell 2** to upload your `kaggle.json` token and download the dataset.
4. Run all remaining cells in order; the Google Drive export is optional.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q kaggle "albumentations==2.0.8" pycocotools tqdm
print('All packages installed.')

## Cell 2 — Upload Kaggle token and download dataset

Download a legacy `kaggle.json` API credential from https://www.kaggle.com/settings.
Upload it when prompted. An existing credential is reused when this cell is rerun.
The dataset is extracted directly into `/content/football-semantic-segmentation`.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from google.colab import files

credential_dir = Path.home() / '.kaggle'
credential_path = credential_dir / 'kaggle.json'
credential_dir.mkdir(parents=True, exist_ok=True)

if not credential_path.is_file():
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Upload the file named kaggle.json, then rerun this cell.')
    credentials = json.loads(uploaded['kaggle.json'])
    if not credentials.get('username') or not credentials.get('key'):
        raise ValueError('kaggle.json must contain a username and an API key.')
    credential_path.write_bytes(uploaded['kaggle.json'])
    del uploaded, credentials
else:
    print('Using the existing Kaggle credential.')
credential_path.chmod(0o600)

DATA_ROOT = Path('/content/football-semantic-segmentation')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

annotation_files = list(DATA_ROOT.rglob('COCO_Football Pixel.json'))
has_images = any(DATA_ROOT.rglob('*.jpg'))
if not annotation_files or not has_images:
    print('Downloading dataset...')
    subprocess.run(
        [
            'kaggle', 'datasets', 'download',
            'sadhliroomyprime/football-semantic-segmentation',
            '-p', str(DATA_ROOT), '--unzip', '--force',
        ],
        check=True,
    )
else:
    print('Dataset already downloaded.')

print('\nFolder structure:')
for root, dirs, files_list in os.walk(DATA_ROOT):
    depth = len(Path(root).relative_to(DATA_ROOT).parts)
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    for filename in sorted(files_list)[:5]:
        print(f'{indent}  {filename}')
    if len(files_list) > 5:
        print(f'{indent}  ... ({len(files_list)} files total)')
    if depth >= 2:
        dirs[:] = []

## Cell 3 — Verify GPU and imports

In [ ]:
import random

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pycocotools.coco import COCO
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__} | Albumentations: {A.__version__}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected. Enable T4 GPU in the Colab runtime settings for training.')

## Cell 4 — Dataset classes and mask conversion

Read category IDs and names from COCO. Build semantic class masks directly from
the annotation geometry, avoiding assumptions about the exported mask colors.
Background is index 0. If annotations overlap, smaller regions are painted last
so that small objects are not hidden by larger regions.

In [ ]:
annotation_files = sorted(DATA_ROOT.rglob('COCO_Football Pixel.json'))
if len(annotation_files) != 1:
    raise FileNotFoundError(
        f'Expected one COCO_Football Pixel.json under {DATA_ROOT}; '
        f'found {len(annotation_files)}. Check the download in Cell 2.'
    )

DATASET_DIR = annotation_files[0].parent
annotation_api = COCO(str(annotation_files[0]))
categories = sorted(annotation_api.loadCats(annotation_api.getCatIds()), key=lambda category: category['id'])
image_records = sorted(annotation_api.loadImgs(annotation_api.getImgIds()), key=lambda image: image['id'])
IMAGE_FILENAMES = [image['file_name'] for image in image_records]
if not categories or not image_records:
    raise ValueError('The COCO file must contain categories and images.')

BACKGROUND_INDEX = 0
CLASS_NAMES = ['Background'] + [category['name'] for category in categories]
CATEGORY_IDS = [None] + [category['id'] for category in categories]
CATEGORY_TO_INDEX = {category_id: index for index, category_id in enumerate(CATEGORY_IDS) if category_id is not None}
NUM_CLASSES = len(CLASS_NAMES)
if NUM_CLASSES > 256:
    raise ValueError('Class-index PNGs require at most 256 classes.')
COLOR_MAP = np.rint(
    plt.get_cmap('tab20', NUM_CLASSES)(np.arange(NUM_CLASSES))[:, :3] * 255
).astype(np.uint8)
COLOR_MAP[BACKGROUND_INDEX] = 0


def coco_mask_to_index(image_record: dict) -> np.ndarray:
    """Rasterize COCO segmentation annotations into contiguous uint8 class indices."""
    index_mask = np.full(
        (image_record['height'], image_record['width']), BACKGROUND_INDEX, dtype=np.uint8,
    )
    annotations = annotation_api.loadAnns(annotation_api.getAnnIds(imgIds=[image_record['id']]))
    for annotation in sorted(annotations, key=lambda item: (-item['area'], item['id'])):
        if annotation['category_id'] not in CATEGORY_TO_INDEX:
            raise ValueError(f'Unknown COCO category ID: {annotation["category_id"]}')
        region = annotation_api.annToMask(annotation).astype(bool)
        index_mask[region] = CATEGORY_TO_INDEX[annotation['category_id']]
    return index_mask


print(f'Number of classes: {NUM_CLASSES}')
for class_index, (name, category_id) in enumerate(zip(CLASS_NAMES, CATEGORY_IDS)):
    print(f'{class_index:2}: {name:20} COCO category ID: {category_id}')

## Cell 5 — Dataset preview

Run this to verify the images and masks loaded correctly before training.

In [ ]:
img_dir = DATASET_DIR / 'images'
MASK_CACHE_DIR = DATA_ROOT / 'class_masks'
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
missing_images = [filename for filename in IMAGE_FILENAMES if not (img_dir / filename).is_file()]
if missing_images:
    raise FileNotFoundError(
        f'Missing {len(missing_images)} COCO images. Examples: {missing_images[:3]}. '
        'Check the download in Cell 2.'
    )

# Rebuild the cache on reruns so it always matches the current category mapping.
samples = []
for image_record in tqdm(image_records, desc='Build semantic masks'):
    img_path = img_dir / image_record['file_name']
    mask_path = MASK_CACHE_DIR / f'{image_record["id"]}.png'
    Image.fromarray(coco_mask_to_index(image_record)).save(mask_path)
    samples.append((img_path, mask_path))
if not samples:
    raise FileNotFoundError(f'No image-mask pairs found in {img_dir}.')

print(f'Image-mask pairs found: {len(samples)}')
preview_count = min(3, len(samples))
fig, axes = plt.subplots(preview_count, 3, figsize=(13, 4 * preview_count), squeeze=False)
for row, (img_path, mask_path) in enumerate(samples[:preview_count]):
    with Image.open(img_path) as image_file, Image.open(mask_path) as mask_file:
        img = np.array(image_file.convert('RGB'))
        mask_idx = np.array(mask_file, dtype=np.uint8)
    if img.shape[:2] != mask_idx.shape:
        raise ValueError(f'Image and mask sizes do not match: {img_path.name}.')

    axes[row, 0].imshow(img)
    axes[row, 0].set_title('Image')
    axes[row, 1].imshow(COLOR_MAP[mask_idx])
    axes[row, 1].set_title('COCO semantic mask')
    axes[row, 2].imshow(img)
    axes[row, 2].imshow(COLOR_MAP[mask_idx], alpha=0.45)
    axes[row, 2].set_title('Mask overlay')
    for ax in axes[row]:
        ax.axis('off')

legend_patches = [plt.Rectangle((0, 0), 1, 1, color=color / 255) for color in COLOR_MAP]
fig.legend(legend_patches, CLASS_NAMES, loc='lower center', ncol=6, fontsize=9)
fig.suptitle('Dataset preview', fontsize=14, fontweight='bold')
plt.tight_layout(rect=(0, 0.10, 1, 0.97))
plt.show()

## Cell 6 — Dataset class and transforms

In [ ]:
IMG_SIZE = 256
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, mask_interpolation=cv2.INTER_NEAREST),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.2),
    A.Affine(
        translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-10, 10),
        border_mode=cv2.BORDER_CONSTANT, fill=0,
        fill_mask=BACKGROUND_INDEX, mask_interpolation=cv2.INTER_NEAREST, p=0.4,
    ),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
], seed=SEED, strict=True)

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, mask_interpolation=cv2.INTER_NEAREST),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
], strict=True)


class FootballSegDataset(Dataset):
    """Load images and cached COCO class masks with synchronized transformations."""

    def __init__(self, samples: list[tuple[Path, Path]], transform=None):
        self.samples = tuple(samples)
        self.transform = transform
        if not self.samples:
            raise ValueError('The dataset must contain at least one image-mask pair.')

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_path, mask_path = self.samples[index]
        with Image.open(img_path) as image_file, Image.open(mask_path) as mask_file:
            image = np.array(image_file.convert('RGB'), dtype=np.uint8)
            mask = np.array(mask_file, dtype=np.uint8)
        if image.shape[:2] != mask.shape:
            raise ValueError(f'Image and mask sizes do not match: {img_path.name}.')
        if np.any(mask >= NUM_CLASSES):
            raise ValueError(f'Mask has an invalid class index: {mask_path.name}. Rebuild masks in Cell 5.')
        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1).copy()).float() / 255.0
            mask = torch.from_numpy(mask)
        # OpenCV receives uint8 masks; CrossEntropyLoss and one_hot require int64.
        return image, mask.long()


print('Dataset class and transforms defined.')

## Cell 7 — U-Net model

In [ ]:
def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = double_conv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = double_conv(2 * out_ch, out_ch)  # skip + up channels

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:             # fix odd spatial dimensions
            x = F.interpolate(x, size=skip.shape[2:],
                              mode='bilinear', align_corners=False)
        x = torch.cat([skip, x], dim=1)       # channel-wise concat
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES,
                 features=(64, 128, 256, 512)):
        super().__init__()
        self.encoders = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(EncoderBlock(ch, f))
            ch = f
        self.bottleneck = double_conv(ch, ch * 2)
        ch = ch * 2
        self.decoders = nn.ModuleList()
        for f in reversed(features):
            self.decoders.append(DecoderBlock(ch, f))
            ch = f
        self.head = nn.Conv2d(ch, num_classes, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            x, skip = enc(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.head(x)   # (B, num_classes, H, W) raw logits


# A forward-only shape check; no graph or BatchNorm state updates.
model = UNet().to(DEVICE)
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = model(dummy)
    assert out.shape == (1, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {out.shape}')
print(f'Parameters   : {total_params:,}')
del dummy, out, model

## Cell 8 — Loss function and metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs  = F.softmax(logits.float(), dim=1)
        onehot = F.one_hot(targets, num_classes).permute(0,3,1,2).float()
        inter  = (probs * onehot).sum(dim=(2,3))
        union  = probs.sum(dim=(2,3)) + onehot.sum(dim=(2,3))
        dice   = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    """0.5 × CrossEntropy  +  0.5 × Dice"""
    def __init__(self):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return 0.5 * self.ce(logits, targets) + 0.5 * self.dice(logits, targets)


def compute_iou(confusion_matrix: torch.Tensor) -> dict[str, float]:
    """Compute per-class and mean IoU from counts over the full validation set."""
    counts = confusion_matrix.to(dtype=torch.float64)
    intersection = counts.diag()
    union = counts.sum(dim=0) + counts.sum(dim=1) - intersection
    ious = {}
    for class_index, name in enumerate(CLASS_NAMES):
        ious[name] = (
            (intersection[class_index] / union[class_index]).item()
            if union[class_index] > 0 else float('nan')
        )
    valid = [value for value in ious.values() if not np.isnan(value)]
    ious['mIoU'] = float(np.mean(valid)) if valid else 0.0
    return ious


print('Loss and metrics defined.')

## Cell 9 — Training and validation functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    """Train for one epoch and return loss averaged over samples."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    for images, masks in tqdm(loader, desc='  Train', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)
    if total_samples == 0:
        raise ValueError('The training loader is empty.')
    return total_loss / total_samples


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Evaluate all batches without retaining predictions for the entire dataset."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    confusion_matrix = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    for images, masks in tqdm(loader, desc='  Val  ', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, masks)
        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)
        predictions = logits.argmax(dim=1)
        encoded = (masks.reshape(-1) * NUM_CLASSES + predictions.reshape(-1)).cpu()
        confusion_matrix += torch.bincount(
            encoded, minlength=NUM_CLASSES ** 2,
        ).reshape(NUM_CLASSES, NUM_CLASSES)
    if total_samples == 0:
        raise ValueError('The validation loader is empty.')
    return total_loss / total_samples, compute_iou(confusion_matrix)


print('Training functions defined.')

## Cell 10 — Build data loaders

Training and validation use distinct dataset instances, so assigning validation
transforms cannot overwrite training augmentation. The split and shuffle are seeded.
Validation metrics include Background and exclude classes absent from both the
ground truth and predictions across the whole validation set.

In [ ]:
BATCH_SIZE = 8
VAL_SPLIT = 0.2
# A single-process loader keeps the seeded augmentation sequence reproducible.
NUM_WORKERS = 0

if len(samples) < 2:
    raise ValueError('At least two image-mask pairs are required for a train/validation split.')
if not 0 < VAL_SPLIT < 1:
    raise ValueError('VAL_SPLIT must be strictly between 0 and 1.')
if BATCH_SIZE < 1:
    raise ValueError('BATCH_SIZE must be positive.')

n_val = min(len(samples) - 1, max(1, int(len(samples) * VAL_SPLIT)))
n_train = len(samples) - n_val
split_indices = torch.randperm(len(samples), generator=torch.Generator().manual_seed(SEED)).tolist()
val_indices = split_indices[:n_val]
train_indices = split_indices[n_val:]

train_set = FootballSegDataset([samples[index] for index in train_indices], transform=train_transform)
val_set = FootballSegDataset([samples[index] for index in val_indices], transform=val_transform)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'), generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
)

print(f'Train samples : {n_train}')
print(f'Val samples   : {n_val}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

## Cell 11 — Training

The best checkpoint is saved to `/content/unet_football_best.pth`, including when
the first validation mIoU is zero. The checkpoint also records the class mapping,
input size, split indices, optimizer, scheduler, and mixed precision scaler state.
Use Cell 14 or Cell 15 to export the checkpoint and plots after training.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
NUM_EPOCHS = 40
LR         = 1e-3
SAVE_PATH  = '/content/unet_football_best.pth'

# ── Reinitialise model, optimizer, scheduler ──────────────────────────────────
torch.manual_seed(SEED)
if NUM_EPOCHS < 1:
    raise ValueError("NUM_EPOCHS must be positive.")
model     = UNet(in_channels=3, num_classes=NUM_CLASSES).to(DEVICE)
criterion = CombinedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

print(f'Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# ── Training loop ──────────────────────────────────────────────────────────────
train_losses, val_losses, val_mious = [], [], []
best_miou = float('-inf')

for epoch in range(1, NUM_EPOCHS + 1):
    lr_now = optimizer.param_groups[0]['lr']
    print(f'\nEpoch {epoch:>2}/{NUM_EPOCHS}  (lr={lr_now:.2e})')

    train_loss             = train_one_epoch(model, train_loader, optimizer,
                                             criterion, DEVICE, scaler)
    val_loss, iou_dict     = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    miou = iou_dict['mIoU']
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_mious.append(miou)

    print(f'  Train loss : {train_loss:.4f}  |  Val loss : {val_loss:.4f}  |  mIoU : {miou:.4f}')
    print('  Per-class IoU:')
    for name, iou in iou_dict.items():
        if name != 'mIoU':
            bar = '█' * int(iou * 20) if not np.isnan(iou) else ''
            print(f'    {name:15s}: {iou:.4f}  {bar}')

    if miou > best_miou:
        best_miou = miou
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'best_miou': best_miou,
            'num_classes': NUM_CLASSES,
            'class_names': CLASS_NAMES,
            'category_ids': CATEGORY_IDS,
            'display_colors': COLOR_MAP.tolist(),
            'img_size': IMG_SIZE,
            'seed': SEED,
            'train_indices': train_indices,
            'val_indices': val_indices,
            'image_filenames': IMAGE_FILENAMES,
        }, SAVE_PATH)
        print(f'  ✓ Best mIoU saved: {best_miou:.4f} → {SAVE_PATH}')

print(f'\nTraining complete. Best mIoU: {best_miou:.4f}')

## Cell 12 — Training curves

In [ ]:
epochs = range(1, len(train_losses) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, train_losses, 'b-o', markersize=4, label='Train loss')
ax1.plot(epochs, val_losses,   'r-o', markersize=4, label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss curves'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, val_mious, 'g-o', markersize=4, label='Val mIoU')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU')
ax2.set_title('Validation mIoU'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13 — Predictions visualization

Load the best checkpoint and display samples drawn only from the validation split.
Ground truth and predictions use the display palette from Cell 4.

In [ ]:
def denormalize(tensor):
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.asarray(IMAGENET_STD) + np.asarray(IMAGENET_MEAN)
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)


# Load best checkpoint
if not Path(SAVE_PATH).is_file():
    raise FileNotFoundError('No checkpoint found. Complete training in Cell 11 first.')
checkpoint = torch.load(SAVE_PATH, map_location='cpu', weights_only=True)
if checkpoint['class_names'] != CLASS_NAMES or checkpoint['category_ids'] != CATEGORY_IDS:
    raise ValueError('The checkpoint class mapping differs from the loaded dataset.')
model.load_state_dict(checkpoint['model_state'])
print(f"Loaded best model from epoch {checkpoint['epoch']} "
      f"(mIoU={checkpoint['best_miou']:.4f})")

# Visualize 4 random validation samples
vis_dataset = val_set
model.eval()

N = min(4, len(vis_dataset))
indices = np.random.default_rng(SEED).choice(len(vis_dataset), N, replace=False)
fig, axes = plt.subplots(N, 3, figsize=(13, N * 4), squeeze=False)

with torch.no_grad():
    for row, idx in enumerate(indices):
        image, mask = vis_dataset[idx]
        logits = model(image.unsqueeze(0).to(DEVICE))
        pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        axes[row, 0].imshow(denormalize(image))
        axes[row, 0].set_title('Image', fontsize=11)
        axes[row, 1].imshow(COLOR_MAP[mask.numpy()])
        axes[row, 1].set_title('Ground truth', fontsize=11)
        axes[row, 2].imshow(COLOR_MAP[pred])
        axes[row, 2].set_title('Prediction', fontsize=11)
        for ax in axes[row]: ax.axis('off')

# Class legend
legend_patches = [plt.Rectangle((0,0),1,1, color=np.array(c)/255) for c in COLOR_MAP]
fig.legend(legend_patches, CLASS_NAMES, loc='lower center',
           ncol=6, fontsize=9)
plt.suptitle('U-Net — Football Segmentation Predictions', fontsize=14, fontweight='bold')
plt.tight_layout(rect=(0, 0.10, 1, 0.97))
plt.savefig('/content/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 14 — Save results to Google Drive (optional)

Run this cell to persist your checkpoint and plots to Google Drive so they survive after the Colab session ends.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/unet_football'
os.makedirs(DRIVE_DIR, exist_ok=True)

for fname in ['unet_football_best.pth', 'training_curves.png', 'predictions.png']:
    src = f'/content/{fname}'
    dst = f'{DRIVE_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {fname} → {dst}')
    else:
        print(f'File not found: {src}')

print('Done! Files saved to Google Drive.')

## Cell 15 — Download checkpoint to local machine (alternative)

In [ ]:
from google.colab import files
files.download('/content/unet_football_best.pth')
files.download('/content/training_curves.png')
files.download('/content/predictions.png')